In [3]:
# Cell 1 — Load processed data
from pathlib import Path
import pandas as pd
import numpy as np
import time

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

DATA_DIR = Path("../data/processed")
RESULTS_DIR = Path("../results/metrics")
MODELS_DIR = Path("../models")

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

X_train = pd.read_csv(DATA_DIR / "X_train_processed.csv")
X_test = pd.read_csv(DATA_DIR / "X_test_processed.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (472432, 421)
X_test: (118108, 421)
y_train: (472432,)
y_test: (118108,)


In [5]:
# Cell 2 — Define evaluation function

def evaluate_model(model_name, model, X_test, y_test):
    start_pred = time.time()
    y_pred = model.predict(X_test)
    prediction_time = time.time() - start_pred

    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)[:, 1]
    else:
        y_proba = y_pred

    results = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_proba),
        "pr_auc": average_precision_score(y_test, y_proba),
        "prediction_time_seconds": prediction_time
    }

    print(f"\n{model_name} Results")
    print("-" * 40)
    for key, value in results.items():
        if key != "model":
            print(f"{key}: {value:.4f}")

    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, zero_division=0))

    return results

In [6]:
# Cell 3 — Train Logistic Regression baseline

log_reg = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

start_train = time.time()
log_reg.fit(X_train, y_train)
training_time = time.time() - start_train

log_reg_results = evaluate_model("Logistic Regression", log_reg, X_test, y_test)
log_reg_results["training_time_seconds"] = training_time

print("Training time:", training_time)

/Users/user/Documents/trustlens/venv312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)
/Users/user/Documents/trustlens/venv312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



Logistic Regression Results
----------------------------------------
accuracy: 0.7299
precision: 0.0795
recall: 0.6354
f1_score: 0.1413
roc_auc: 0.7497
pr_auc: 0.1298
prediction_time_seconds: 0.9581

Confusion Matrix:
[[83578 30397]
 [ 1507  2626]]

Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.73      0.84    113975
           1       0.08      0.64      0.14      4133

    accuracy                           0.73    118108
   macro avg       0.53      0.68      0.49    118108
weighted avg       0.95      0.73      0.82    118108

Training time: 257.9099311828613


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

log_reg_scaled = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42
    ))
])

start_train = time.time()
log_reg_scaled.fit(X_train, y_train)
training_time = time.time() - start_train

log_reg_scaled_results = evaluate_model(
    "Logistic Regression (Scaled)",
    log_reg_scaled,
    X_test,
    y_test
)

log_reg_scaled_results["training_time_seconds"] = training_time

print("Training time:", training_time)

/Users/user/Documents/trustlens/venv312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1457: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


In [8]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    min_samples_split=10,
    min_samples_leaf=5,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=42
)

start_train = time.time()
rf_model.fit(X_train, y_train)
training_time = time.time() - start_train

rf_results = evaluate_model(
    "Random Forest",
    rf_model,
    X_test,
    y_test
)

rf_results["training_time_seconds"] = training_time

print("Training time:", training_time)


Random Forest Results
----------------------------------------
accuracy: 0.8973
precision: 0.2162
recall: 0.7372
f1_score: 0.3344
roc_auc: 0.8986
pr_auc: 0.5609
prediction_time_seconds: 1.4626

Confusion Matrix:
[[102931  11044]
 [  1086   3047]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.90      0.94    113975
           1       0.22      0.74      0.33      4133

    accuracy                           0.90    118108
   macro avg       0.60      0.82      0.64    118108
weighted avg       0.96      0.90      0.92    118108

Training time: 75.42487406730652


In [12]:
model_results = []

model_results.append(log_reg_results)
model_results.append(rf_results)
model_results.append(lgbm_results)
model_results.append(xgb_results)


results_df = pd.DataFrame(model_results)
results_df.to_csv(RESULTS_DIR / "model_results_initial.csv", index=False)

results_df

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc,prediction_time_seconds,training_time_seconds
0,Logistic Regression,0.729874,0.079520,0.635374,0.141350,0.749713,0.129822,0.958090,257.909931
1,Random Forest,0.897297,0.216237,0.737237,0.334394,0.898557,0.560873,1.462579,75.424874
2,LightGBM,0.923782,0.294574,0.844665,0.436812,0.954520,0.720850,2.636820,45.743817
3,XGBoost,0.906509,0.247423,0.818776,0.380011,0.937409,0.655553,0.749893,87.771222


In [10]:
from lightgbm import LGBMClassifier

# Calculate imbalance ratio for scale_pos_weight
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    n_jobs=-1
)

start_train = time.time()
lgbm_model.fit(X_train, y_train)
training_time = time.time() - start_train

lgbm_results = evaluate_model(
    "LightGBM",
    lgbm_model,
    X_test,
    y_test
)

lgbm_results["training_time_seconds"] = training_time

print("Training time:", training_time)

[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.716222 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 37873
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 419
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101

LightGBM Results
----------------------------------------
accuracy: 0.9238
precision: 0.2946
recall: 0.8447
f1_score: 0.4368
roc_auc: 0.9545
pr_auc: 0.7208
prediction_time_seconds: 2.6368

Confusion Matrix:
[[105615   8360]
 [   642   3491]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.93      0.96    113975
           1       0.29      0.84      0.44      4133

    acc

In [11]:
from xgboost import XGBClassifier

neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

start_train = time.time()
xgb_model.fit(X_train, y_train)
training_time = time.time() - start_train

xgb_results = evaluate_model(
    "XGBoost",
    xgb_model,
    X_test,
    y_test
)

xgb_results["training_time_seconds"] = training_time

print("Training time:", training_time)


XGBoost Results
----------------------------------------
accuracy: 0.9065
precision: 0.2474
recall: 0.8188
f1_score: 0.3800
roc_auc: 0.9374
pr_auc: 0.6556
prediction_time_seconds: 0.7499

Confusion Matrix:
[[103682  10293]
 [   749   3384]]

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.91      0.95    113975
           1       0.25      0.82      0.38      4133

    accuracy                           0.91    118108
   macro avg       0.62      0.86      0.66    118108
weighted avg       0.97      0.91      0.93    118108

Training time: 87.77122187614441


In [4]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data/processed")

X_train = pd.read_csv(DATA_DIR / "X_train_processed.csv")
X_test = pd.read_csv(DATA_DIR / "X_test_processed.csv")
y_train = pd.read_csv(DATA_DIR / "y_train.csv").squeeze()
y_test = pd.read_csv(DATA_DIR / "y_test.csv").squeeze()

print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(472432, 421)
(118108, 421)
(472432,)
(118108,)


In [6]:
print(y_train.value_counts())

isFraud
0    455902
1     16530
Name: count, dtype: int64


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import time

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

In [13]:
import joblib
from pathlib import Path

MODELS_DIR = Path("../models")
RESULTS_DIR = Path("../results/metrics")

MODELS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(log_reg, MODELS_DIR / "logistic_regression.pkl")
joblib.dump(rf_model, MODELS_DIR / "random_forest.pkl")
joblib.dump(lgbm_model, MODELS_DIR / "lightgbm.pkl")
joblib.dump(xgb_model, MODELS_DIR / "xgboost.pkl")

results_df.to_csv(RESULTS_DIR / "model_results_final.csv", index=False)

print("Models and final results saved successfully.")

Models and final results saved successfully.


In [15]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.metrics import matthews_corrcoef

# Find project root whether notebook is opened from /trustlens or /trustlens/notebooks
ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

# Load processed test data
X_test = pd.read_csv(ROOT_DIR / "data" / "processed" / "X_test_processed.csv")
y_test = pd.read_csv(ROOT_DIR / "data" / "processed" / "y_test.csv").squeeze()

# Saved models
model_files = {
    "Logistic Regression": ROOT_DIR / "models" / "logistic_regression.pkl",
    "Random Forest": ROOT_DIR / "models" / "random_forest.pkl",
    "LightGBM": ROOT_DIR / "models" / "lightgbm.pkl",
    "XGBoost": ROOT_DIR / "models" / "xgboost.pkl",
}

mcc_results = []

for model_name, model_path in model_files.items():
    model = joblib.load(model_path)

    # Use same default 0.5 threshold
    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)
    else:
        y_pred = model.predict(X_test)

    mcc = matthews_corrcoef(y_test, y_pred)

    mcc_results.append({
        "model": model_name,
        "mcc": mcc
    })

mcc_df = pd.DataFrame(mcc_results)

mcc_df

,model,mcc
0,Logistic Regression,0.150955
1,Random Forest,0.363015
2,LightGBM,0.471751
3,XGBoost,0.418347


In [16]:
from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

mcc_df = pd.DataFrame(mcc_results)

results_files = [
    ROOT_DIR / "results" / "metrics" / "model_results_final.csv",
    ROOT_DIR / "results" / "metrics" / "trustlens_model_comparison_table.csv",
]

for results_path in results_files:
    if results_path.exists():
        results_df = pd.read_csv(results_path)

        # Remove old MCC column if it already exists
        results_df = results_df.drop(columns=["mcc"], errors="ignore")

        # Add new MCC values
        results_df = results_df.merge(mcc_df, on="model", how="left")

        # Save updated file
        results_df.to_csv(results_path, index=False)

        print(f"Updated: {results_path}")
        display(results_df)
    else:
        print(f"File not found: {results_path}")

Updated: /Users/user/Documents/trustlens/results/metrics/model_results_final.csv


,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc,prediction_time_seconds,training_time_seconds,mcc
0,Logistic Regression,0.729874,0.079520,0.635374,0.141350,0.749713,0.129822,0.958090,257.909931,0.150955
1,Random Forest,0.897297,0.216237,0.737237,0.334394,0.898557,0.560873,1.462579,75.424874,0.363015
2,LightGBM,0.923782,0.294574,0.844665,0.436812,0.954520,0.720850,2.636820,45.743817,0.471751
3,XGBoost,0.906509,0.247423,0.818776,0.380011,0.937409,0.655553,0.749893,87.771222,0.418347


Updated: /Users/user/Documents/trustlens/results/metrics/trustlens_model_comparison_table.csv


,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc,training_time_seconds,prediction_time_seconds,interpretation_method,shap_explanation_time_500_samples,mcc
0,Logistic Regression,0.729874,0.079520,0.635374,0.141350,0.749713,0.129822,257.909931,0.958090,Model Coefficients,NaN,0.150955
1,Random Forest,0.897297,0.216237,0.737237,0.334394,0.898557,0.560873,75.424874,1.462579,SHAP,27.573582,0.363015
2,LightGBM,0.923782,0.294574,0.844665,0.436812,0.954520,0.720850,45.743817,2.636820,SHAP,3.396310,0.471751
3,XGBoost,0.906509,0.247423,0.818776,0.380011,0.937409,0.655553,87.771222,0.749893,SHAP,0.881750,0.418347


In [17]:
from pathlib import Path
import joblib
import pandas as pd

# Find project root
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

# Load the exact saved models used in your experiments
model_files = {
    "Logistic Regression": ROOT_DIR / "models" / "logistic_regression.pkl",
    "Random Forest": ROOT_DIR / "models" / "random_forest.pkl",
    "LightGBM": ROOT_DIR / "models" / "lightgbm.pkl",
    "XGBoost": ROOT_DIR / "models" / "xgboost.pkl",
}

models = {
    model_name: joblib.load(model_path)
    for model_name, model_path in model_files.items()
}

# Print the exact parameters stored in each model
for model_name, model in models.items():
    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)

    params = model.get_params()

    for parameter, value in sorted(params.items()):
        print(f"{parameter}: {value}")


Logistic Regression
C: 1.0
class_weight: balanced
dual: False
fit_intercept: True
intercept_scaling: 1
l1_ratio: 0.0
max_iter: 1000
n_jobs: -1
penalty: deprecated
random_state: 42
solver: lbfgs
tol: 0.0001
verbose: 0
warm_start: False

Random Forest
bootstrap: True
ccp_alpha: 0.0
class_weight: balanced_subsample
criterion: gini
max_depth: 12
max_features: sqrt
max_leaf_nodes: None
max_samples: None
min_impurity_decrease: 0.0
min_samples_leaf: 5
min_samples_split: 10
min_weight_fraction_leaf: 0.0
monotonic_cst: None
n_estimators: 100
n_jobs: -1
oob_score: False
random_state: 42
verbose: 0
warm_start: False

LightGBM
boosting_type: gbdt
class_weight: None
colsample_bytree: 0.8
importance_type: split
learning_rate: 0.05
max_depth: -1
min_child_samples: 20
min_child_weight: 0.001
min_split_gain: 0.0
n_estimators: 300
n_jobs: -1
num_leaves: 64
objective: None
random_state: 42
reg_alpha: 0.0
reg_lambda: 0.0
scale_pos_weight: 27.580278281911674
subsample: 0.8
subsample_for_bin: 200000
subsam

In [18]:
important_parameters = {
    "Logistic Regression": [
        "C",
        "class_weight",
        "max_iter",
        "solver",
        "random_state"
    ],

    "Random Forest": [
        "n_estimators",
        "max_depth",
        "min_samples_split",
        "min_samples_leaf",
        "max_features",
        "class_weight",
        "random_state",
        "n_jobs"
    ],

    "LightGBM": [
        "n_estimators",
        "learning_rate",
        "num_leaves",
        "max_depth",
        "class_weight",
        "scale_pos_weight",
        "random_state",
        "n_jobs"
    ],

    "XGBoost": [
        "n_estimators",
        "learning_rate",
        "max_depth",
        "subsample",
        "colsample_bytree",
        "scale_pos_weight",
        "random_state",
        "n_jobs"
    ],
}

rows = []

for model_name, model in models.items():
    params = model.get_params()

    row = {
        "Model": model_name
    }

    for parameter in important_parameters[model_name]:
        row[parameter] = params.get(parameter)

    rows.append(row)

configuration_df = pd.DataFrame(rows)

display(configuration_df)

,Model,C,class_weight,max_iter,solver,random_state,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,n_jobs,learning_rate,num_leaves,scale_pos_weight,subsample,colsample_bytree
0,Logistic Regression,1.0,balanced,1000.0,lbfgs,42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Random Forest,NaN,balanced_subsample,NaN,NaN,42,100.0,12.0,10.0,5.0,sqrt,-1.0,NaN,NaN,NaN,NaN,NaN
2,LightGBM,NaN,NaN,NaN,NaN,42,300.0,-1.0,NaN,NaN,NaN,-1.0,0.05,64.0,27.580278,NaN,NaN
3,XGBoost,NaN,NaN,NaN,NaN,42,300.0,6.0,NaN,NaN,NaN,-1.0,0.05,NaN,27.580278,0.8,0.8


In [19]:
output_path = (
    ROOT_DIR
    / "results"
    / "metrics"
    / "model_configuration.csv"
)

configuration_df.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: /Users/user/Documents/trustlens/results/metrics/model_configuration.csv


In [20]:
import pandas as pd

X_train_check = pd.read_csv(
    ROOT_DIR / "data" / "processed" / "X_train_processed.csv"
)

print("TransactionID included:", "TransactionID" in X_train_check.columns)
print("Number of features:", X_train_check.shape[1])

TransactionID included: True
Number of features: 421


In [23]:
from pathlib import Path
import pandas as pd

ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

# Load the existing processed datasets
X_train = pd.read_csv(
    ROOT_DIR / "data" / "processed" / "X_train_processed.csv"
)

X_test = pd.read_csv(
    ROOT_DIR / "data" / "processed" / "X_test_processed.csv"
)

y_train = pd.read_csv(
    ROOT_DIR / "data" / "processed" / "y_train.csv"
).squeeze()

y_test = pd.read_csv(
    ROOT_DIR / "data" / "processed" / "y_test.csv"
).squeeze()

print("Before correction")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("TransactionID in training data:", "TransactionID" in X_train.columns)

# Remove the identifier from the predictive feature set
X_train_clean = X_train.drop(columns=["TransactionID"])
X_test_clean = X_test.drop(columns=["TransactionID"])

print("\nAfter correction")
print("X_train_clean:", X_train_clean.shape)
print("X_test_clean:", X_test_clean.shape)
print(
    "TransactionID in corrected data:",
    "TransactionID" in X_train_clean.columns
)

Before correction
X_train: (472432, 421)
X_test: (118108, 421)
TransactionID in training data: True

After correction
X_train_clean: (472432, 420)
X_test_clean: (118108, 420)
TransactionID in corrected data: False


In [24]:
X_train_clean.to_csv(
    ROOT_DIR / "data" / "processed" / "X_train_processed_no_id.csv",
    index=False
)

X_test_clean.to_csv(
    ROOT_DIR / "data" / "processed" / "X_test_processed_no_id.csv",
    index=False
)

print("Corrected datasets saved.")

Corrected datasets saved.


In [25]:
import time
import joblib
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    matthews_corrcoef,
)

models_clean = {
    "Logistic Regression": LogisticRegression(
        C=1.0,
        class_weight="balanced",
        max_iter=1000,
        solver="lbfgs",
        random_state=42,
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        min_samples_split=10,
        min_samples_leaf=5,
        max_features="sqrt",
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1,
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=64,
        max_depth=-1,
        scale_pos_weight=27.580278,
        random_state=42,
        n_jobs=-1,
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=27.580278,
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss",
    ),
}

In [26]:
corrected_results = []

for model_name, model in models_clean.items():

    print(f"\nTraining {model_name}...")

    # Training time
    start_time = time.perf_counter()

    model.fit(X_train_clean, y_train)

    training_time = time.perf_counter() - start_time

    # Prediction time
    start_time = time.perf_counter()

    y_prob = model.predict_proba(X_test_clean)[:, 1]

    prediction_time = time.perf_counter() - start_time

    # Same 0.50 threshold used in the original experiment
    y_pred = (y_prob >= 0.5).astype(int)

    results = {
        "model": model_name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1_score": f1_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "pr_auc": average_precision_score(y_test, y_prob),
        "mcc": matthews_corrcoef(y_test, y_pred),
        "training_time_seconds": training_time,
        "prediction_time_seconds": prediction_time,
    }

    corrected_results.append(results)

    print(results)


Training Logistic Regression...


/Users/user/Documents/trustlens/venv312/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


{'model': 'Logistic Regression', 'accuracy': 0.664662850949978, 'precision': 0.06947107799111586, 'recall': 0.692475199612872, 'f1_score': 0.12627399073461285, 'roc_auc': 0.7521289168488406, 'pr_auc': 0.1379939929582216, 'mcc': 0.1373151450992272, 'training_time_seconds': 279.49071012303466, 'prediction_time_seconds': 1.1354238239582628}

Training Random Forest...
{'model': 'Random Forest', 'accuracy': 0.8944525349679954, 'precision': 0.21184037623625424, 'recall': 0.7411081538833777, 'f1_score': 0.32949655765920827, 'roc_auc': 0.8985922295136587, 'pr_auc': 0.5610901216567403, 'mcc': 0.35943969052186986, 'training_time_seconds': 81.16854012699332, 'prediction_time_seconds': 1.498031251016073}

Training LightGBM...
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.679907 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `

In [27]:
corrected_results_df = pd.DataFrame(corrected_results)

display(corrected_results_df)

,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc,mcc,training_time_seconds,prediction_time_seconds
0,Logistic Regression,0.664663,0.069471,0.692475,0.126274,0.752129,0.137994,0.137315,279.490710,1.135424
1,Random Forest,0.894453,0.211840,0.741108,0.329497,0.898592,0.561090,0.359440,81.168540,1.498031
2,LightGBM,0.923832,0.294827,0.845391,0.437187,0.954846,0.720748,0.472212,41.225751,2.660839
3,XGBoost,0.905756,0.246008,0.819985,0.378469,0.937218,0.656367,0.417259,87.925848,0.692741


In [28]:
corrected_model_files = {
    "Logistic Regression": "logistic_regression_no_id.pkl",
    "Random Forest": "random_forest_no_id.pkl",
    "LightGBM": "lightgbm_no_id.pkl",
    "XGBoost": "xgboost_no_id.pkl",
}

for model_name, filename in corrected_model_files.items():

    joblib.dump(
        models_clean[model_name],
        ROOT_DIR / "models" / filename
    )

print("Corrected models saved.")

Corrected models saved.


In [29]:
corrected_results_df.to_csv(
    ROOT_DIR
    / "results"
    / "metrics"
    / "model_results_no_transaction_id.csv",
    index=False
)

print("Corrected results saved.")

Corrected results saved.


In [30]:
from pathlib import Path
import shutil

ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

backup_dir = ROOT_DIR / "backup" / "transaction_id_experiment"
backup_dir.mkdir(parents=True, exist_ok=True)

files_to_backup = [
    ROOT_DIR / "models" / "logistic_regression.pkl",
    ROOT_DIR / "models" / "random_forest.pkl",
    ROOT_DIR / "models" / "lightgbm.pkl",
    ROOT_DIR / "models" / "xgboost.pkl",

    ROOT_DIR / "data" / "processed" / "X_train_processed.csv",
    ROOT_DIR / "data" / "processed" / "X_test_processed.csv",

    ROOT_DIR / "results" / "metrics" / "model_results_final.csv",
    ROOT_DIR / "results" / "metrics" / "trustlens_model_comparison_table.csv",
]

for file_path in files_to_backup:
    if file_path.exists():
        destination = backup_dir / file_path.name
        shutil.copy2(file_path, destination)
        print(f"Backed up: {file_path.name}")
    else:
        print(f"Not found: {file_path}")

print(f"\nBackup complete: {backup_dir}")

Backed up: logistic_regression.pkl
Backed up: random_forest.pkl
Backed up: lightgbm.pkl
Backed up: xgboost.pkl
Backed up: X_train_processed.csv
Backed up: X_test_processed.csv
Backed up: model_results_final.csv
Backed up: trustlens_model_comparison_table.csv

Backup complete: /Users/user/Documents/trustlens/backup/transaction_id_experiment


In [31]:
corrected_models = {
    "logistic_regression_no_id.pkl": "logistic_regression.pkl",
    "random_forest_no_id.pkl": "random_forest.pkl",
    "lightgbm_no_id.pkl": "lightgbm.pkl",
    "xgboost_no_id.pkl": "xgboost.pkl",
}

for corrected_name, official_name in corrected_models.items():
    source = ROOT_DIR / "models" / corrected_name
    destination = ROOT_DIR / "models" / official_name

    if source.exists():
        shutil.copy2(source, destination)
        print(f"Promoted: {corrected_name} → {official_name}")
    else:
        print(f"Missing corrected model: {source}")

Promoted: logistic_regression_no_id.pkl → logistic_regression.pkl
Promoted: random_forest_no_id.pkl → random_forest.pkl
Promoted: lightgbm_no_id.pkl → lightgbm.pkl
Promoted: xgboost_no_id.pkl → xgboost.pkl


In [32]:
import joblib

official_lgbm = joblib.load(
    ROOT_DIR / "models" / "lightgbm.pkl"
)

print("Official LightGBM feature count:", official_lgbm.n_features_in_)

Official LightGBM feature count: 420


In [33]:
corrected_datasets = {
    "X_train_processed_no_id.csv": "X_train_processed.csv",
    "X_test_processed_no_id.csv": "X_test_processed.csv",
}

for corrected_name, official_name in corrected_datasets.items():
    source = ROOT_DIR / "data" / "processed" / corrected_name
    destination = ROOT_DIR / "data" / "processed" / official_name

    if source.exists():
        shutil.copy2(source, destination)
        print(f"Promoted: {corrected_name} → {official_name}")
    else:
        print(f"Missing corrected dataset: {source}")

Promoted: X_train_processed_no_id.csv → X_train_processed.csv
Promoted: X_test_processed_no_id.csv → X_test_processed.csv


In [34]:
import pandas as pd

X_train_check = pd.read_csv(
    ROOT_DIR / "data" / "processed" / "X_train_processed.csv",
    nrows=5
)

X_test_check = pd.read_csv(
    ROOT_DIR / "data" / "processed" / "X_test_processed.csv",
    nrows=5
)

print("Training features:", X_train_check.shape[1])
print("Test features:", X_test_check.shape[1])
print("TransactionID in training data:", "TransactionID" in X_train_check.columns)
print("TransactionID in test data:", "TransactionID" in X_test_check.columns)

Training features: 420
Test features: 420
TransactionID in training data: False
TransactionID in test data: False


In [35]:
official_results_path = (
    ROOT_DIR
    / "results"
    / "metrics"
    / "model_results_final.csv"
)

corrected_results_df.to_csv(
    official_results_path,
    index=False
)

print(f"Official corrected results saved to:\n{official_results_path}")

display(corrected_results_df)

Official corrected results saved to:
/Users/user/Documents/trustlens/results/metrics/model_results_final.csv


,model,accuracy,precision,recall,f1_score,roc_auc,pr_auc,mcc,training_time_seconds,prediction_time_seconds
0,Logistic Regression,0.664663,0.069471,0.692475,0.126274,0.752129,0.137994,0.137315,279.490710,1.135424
1,Random Forest,0.894453,0.211840,0.741108,0.329497,0.898592,0.561090,0.359440,81.168540,1.498031
2,LightGBM,0.923832,0.294827,0.845391,0.437187,0.954846,0.720748,0.472212,41.225751,2.660839
3,XGBoost,0.905756,0.246008,0.819985,0.378469,0.937218,0.656367,0.417259,87.925848,0.692741
